### Document Chains Demo
Document Chains allow you to process and analyze lage amount of text data efficiently. They provide a structured approach to working with documents enabling you to retrieve, filter, refine and rank them based on specific criteria.

By using different types of Document Chains like Stuff, Refine, Map Reduce, or Map Re-rank, you can perform specific operations on the retrieved documents and obtain more accurate and relevant results.

In [14]:
import os
import getpass
import textwrap

from langchain_google_genai import GoogleGenerativeAI
from langchain.chains.mapreduce import MapReduceChain
from langchain.prompts import PromptTemplate
from langchain.chains.summarize import load_summarize_chain
from langchain.docstore.document import Document
from langchain.text_splitter import CharacterTextSplitter

from dotenv import load_dotenv

In [5]:
load_dotenv()

True

In [16]:
model=GoogleGenerativeAI(model="gemini-2.0-flash",temperature=0.5)

### Stuff Chain
This revolves putting all relevant data into the Prompt for LangChain's StuffDocumentsChain to process. The advantage of this method is that it only requires one call to the LLM, and the model has access to all the information at once.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("RAG/Resume 2.pdf")
docs = loader.load()

In [10]:
cnt=0
for doc in docs:
    cnt += 1
    print("------ Document #", cnt)
    print(doc.page_content.strip())

------ Document # 1
Data Scientist
SUMMARY
Around 5 years of corporate experience in leveraging advanced analytics and machine learning techniques to drive strategic insights and business growth. 
Skilled in developing predictive models and data-driven solutions using Python and SQL, with a proven track record of improving forecast accuracy and 
operational efficiency. Known for creating interactive dashboards and conducting A/B testing to optimize decision-making processes. Proven ability to 
wrangle and clean data, build and deploy machine learning models, and communicate insights to stakeholders.
SKILLS
Programming Language: Python (Pandas, Numpy,  Scikit-learn, Seaborn, Matplotlib, Plotly, Pytorch)
Machine Learning: Algorithms (supervised, unsupervised, reinforcement learning)
Data Wrangling & Cleaning: Data Manipulation, Data Cleaning techniques, Data Quality control, EDA
Statistics & Modeling: Statistical Analysis, Hypothesis Testing, Time Series analysis, Regression Analysis
Dat

In [11]:
prompt_template = """
You are given a Resume as the below text.
---
{text}
---
Question: Please respond with the Key Skills and Experience summary of the person.
Key Skills:
Experience Summary:
"""

In [17]:
prompt = PromptTemplate(template=prompt_template, input_variables=["text"])
stuff_chain = load_summarize_chain(model, chain_type="stuff", prompt=prompt)
output_summary = stuff_chain.run(docs)

/var/folders/0q/sh8dwyl54mx5k1zrdxhrm8m80000gn/T/ipykernel_50626/2113108531.py:3: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  output_summary = stuff_chain.run(docs)


In [18]:
print(output_summary)

**Key Skills:**

*   Programming Language: Python (Pandas, Numpy, Scikit-learn, Seaborn, Matplotlib, Plotly, Pytorch)
*   Machine Learning: Algorithms (supervised, unsupervised, reinforcement learning)
*   Data Wrangling & Cleaning: Data Manipulation, Data Cleaning techniques, Data Quality control, EDA
*   Statistics & Modeling: Statistical Analysis, Hypothesis Testing, Time Series analysis, Regression Analysis
*   Data Management: SQL, Databases (NoSQL, Relational)
*   Data Visualisation: Tools (Tableau, Excel Visualisation, Matplotlib), Data Storytelling
*   Other Tools: Advanced Excel (HLOOKUPS, VLOOKUPS, Pivots, Conditional Formatting), Git, Bitbucket, JIRA
*   Other Skills: Deep Learning (Computer Vision), Natural Language Processing (NLP), Artificial Intelligence (AI), Predictive Modeling/Analytics

**Experience Summary:**

*   Around 5 years of corporate experience in leveraging advanced analytics and machine learning techniques.
*   Developed predictive models and data-driven s

### Refine Chain
The Refine Chain uses an iterative process to generate a response by analyzing each input document and updating its answers accordingly.
It passes all non-document inputs, the current doscument, and the latest intermediate answer to an LLM chain to obtain a new answer for each document. This chain is ideal for tasks that involve analyzing more documents than can fit in the model's context, as it only passes a single document to an LLM at a time.

In [ ]:
refine_chain = load_summarize_chain(model, chain_type="refine")
print(refine_chain.refine_llm_chain.prompt.template) # In-built prompt for refine chain

Your job is to produce a final summary.
We have provided an existing summary up to a certain point: {existing_answer}
We have the opportunity to refine the existing summary (only if needed) with some more context below.
------------
{text}
------------
Given the new context, refine the original summary.
If the context isn't useful, return the original summary.


In [21]:
output_summary = refine_chain.run(docs)
output_summary

'A data scientist with 5 years of experience leveraging advanced analytics and machine learning (Python, SQL) to drive business growth. Proven ability to build predictive models, improve data quality, create interactive dashboards, and communicate insights. Expertise includes data wrangling, statistical modeling, and data visualization. Holds an MS in Data Science and has experience at Bank of America, where he significantly reduced database space, improved data accuracy, and developed predictive models.'

### Map-Reduce Chain
To process large amount of data efficiently, the MapReduceDocumentsChain method is used.
This involves applying LLM Chain to each document individually (in the Map step), producing a new document. Then all the new documents are passed to a separate combine documents chain to get a single output(in the Reduce step). If necessary, the mapped documments can be compressed before passing them to the combine document chain.
This compression step is performed recursively.

**Major difference betweeen Refine chain MR Chain is Refine Chain is sequential and MR is parallel**

In [23]:
map_reduce_chain = load_summarize_chain(model, chain_type="map_reduce", verbose=True)
print(map_reduce_chain.llm_chain.prompt.template) # In-built prompt for map_reduce chain

Write a concise summary of the following:


"{text}"


CONCISE SUMMARY:


In [24]:
output_summary = map_reduce_chain.run(docs)
print(output_summary)



> Entering new MapReduceDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Write a concise summary of the following:


"Data Scientist
SUMMARY
Around 5 years of corporate experience in leveraging advanced analytics and machine learning techniques to drive strategic insights and business growth. 
Skilled in developing predictive models and data-driven solutions using Python and SQL, with a proven track record of improving forecast accuracy and 
operational efficiency. Known for creating interactive dashboards and conducting A/B testing to optimize decision-making processes. Proven ability to 
wrangle and clean data, build and deploy machine learning models, and communicate insights to stakeholders.
SKILLS
Programming Language: Python (Pandas, Numpy,  Scikit-learn, Seaborn, Matplotlib, Plotly, Pytorch)
Machine Learning: Algorithms (supervised, unsupervised, reinforcement learning)
Data Wrangling & Cleaning: Data Manipulation, Data Cleaning techniques, D